# Sanity de las metricas: runs piloto

Objetivo: comprobar, antes de ningun contraste de hipotesis, si las metricas de gradiente **tienen sentido** sobre las 24 runs del pilot de calibracion. Cuatro diagnosticos, todos apoyados en `src/analysis.py`:

1. **Validez y rangos**: NaN/Inf, columnas ausentes (metrica que fallo en runtime) y valores fuera del rango teorico, mas las identidades duras entre columnas.
2. **Degeneracion**: metricas casi constantes dentro de un run (no aportan senal temporal).
3. **Direccion vs teoria**: si la trayectoria se mueve en el sentido que predice cada paper.
4. **Redundancia**: que metricas se mueven juntas (exploratorio, anticipa la poda de redundantes).

**Alcance.** El pilot tiene **un run por celda**, asi que no hay dispersion intra-celda que correlacionar contra la eficiencia: nada de este notebook toca el plan confirmatorio congelado (`docs/research/Plan de analisis congelado.md`), que corre sobre la matriz en `reports/`. Las mismas funciones serviran luego para la matriz.

**Cautelas del pilot.**
- Las runs corren a **2x presupuesto**, hasta dentro del sobreajuste. `progress_frac` es relativo a ese 2x, asi que el presupuesto congelado 1x equivale a `progress_frac` 0.5. Las predicciones de los papers son sobre la fase de entrenamiento, no sobre la cola post-meseta: por eso la direccion se mira tambien en ventana temprana.
- En `tiny_imagenet` los campos de test/gap del `summary.json` estan corruptos (bug pre-fix); las metricas de la trayectoria y los campos de val/tiempo son validos.


## Setup y carga

In [ ]:
import sys, pathlib
import pandas as pd

# Localiza la raiz del repo (carpeta con pyproject.toml) y anade src/ al path.
_p = pathlib.Path.cwd()
ROOT = next((q for q in [_p, *_p.parents] if (q / "pyproject.toml").exists()), _p)
sys.path.insert(0, str(ROOT / "src"))

import analysis as A      # backend sin ploteo: carga y diagnosticos
import plots as P         # capa de figuras: estilo unico del TFG

P.use_thesis_style()

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 60)

traj = A.load_trajectories()        # por defecto reports_pilot/
summ = A.load_summaries()
print(f"{traj['run_name'].nunique()} runs, {len(traj)} filas epoca-run")
print(f"metricas conocidas: {len(A.metric_columns())} | headline: {len(A.headline_columns())}")

## 1. Validez y rangos

`validity_report` da, por columna conocida: rango observado, conteos de NaN/Inf, valores fuera de cota y estado. `identity_report` comprueba invariantes deterministas que **deben** cumplirse fila a fila (cualquier violacion es un bug de implementacion):

- `eta = -min_cos` (confusion).
- `min_cos <= p05_cos <= median_cos` (ordenacion de cuantiles del mismo set de cosenos).
- `gsnr median <= p95`.
- `tse/cumulative` no decreciente (suma corrida de perdidas no negativas).


In [ ]:
val = A.validity_report(traj)
print("Columnas fuera de 'ok':", (val["status"] != "ok").sum(), "de", len(val))
val

In [ ]:
A.identity_report(traj)

## 2. Degeneracion (informatividad intra-run)

Para cada (run, metrica): `rel_move = within_std / ref`, donde `ref` es la RMS de los `within_std` de esa metrica (su movimiento *tipico* dentro de un run). Se normaliza por esta referencia intra-run y no por el std global a proposito: el std global esta inflado por las diferencias de escala entre datasets (val_loss en mnist ~0.05 vs tiny ~4) y marcaria como planas curvas que claramente se mueven. `rel_move` cercano a 0 = metrica practicamente constante en ese run.


In [ ]:
deg = A.degeneracy_report(traj)
A.degeneracy_summary(deg).round(3)

In [ ]:
deg = A.degeneracy_report(traj)
resumen = A.degeneracy_summary(deg)
fig = P.strip(deg, "rel_move", "key", order=list(resumen.index),
              xlabel="rel_move (0 = constante en ese run)")
P.save(fig, "pilot-degeneracion");
resumen.round(3)

In [ ]:
full = A.trend_summary(A.trend_report(traj))
early = A.trend_summary(A.trend_report(traj, progress_max=0.25))   # ~ mitad del presupuesto 1x
cmp = full[["family", "expected", "frac_agree", "median_rho"]].join(
    early[["frac_agree", "median_rho"]], rsuffix="_early")
cmp.round(3)

In [ ]:
td = A.trend_report(traj)
piv = td.pivot(index="run_name", columns="key", values="rho")
piv = piv[full.index]   # ordena por concordancia
fig = P.heatmap(piv, cbar_label="Spearman(valor, epoca)", vmin=-1, vmax=1)
P.save(fig, "pilot-tendencia");

In [ ]:
# frac_agree por metrica (trayectoria completa vs ventana temprana).
# Se caen las metricas sin prediccion direccional, que no tienen con que concordar.
fig = P.agreement_bars(
    cmp.dropna(subset=["frac_agree", "frac_agree_early"]),
    ("frac_agree", "frac_agree_early"),
    ("trayectoria completa", "ventana temprana"),
    xlabel="fraccion de runs que concuerdan con la teoria",
)
P.save(fig, "pilot-concordancia");

## 4. Redundancia entre metricas (exploratorio)

`redundancy_matrix` promedia las matrices de Spearman intra-run sobre las metricas headline (8 de gradiente + val_acc/val_loss + tse-ema). Promediar dentro de cada run evita que las diferencias de escala entre datasets fabriquen la correlacion (guardia de Simpson). **No es un contraste confirmatorio** ni alimenta ninguna hipotesis: es la foto previa para decidir la poda de metricas redundantes.


In [ ]:
corr = A.redundancy_matrix(traj)
fig = P.heatmap(corr, cbar_label="Spearman intra-run promediado",
                vmin=-1, vmax=1, annot=True)
P.save(fig, "pilot-redundancia");

In [ ]:
A.top_redundant_pairs(corr, n=12).round(3)

## 5. Galeria de trayectorias

Inspeccion visual final: cada metrica headline frente a `progress_frac`, una linea por run, coloreada por dataset. Es el "se mueven como deberian" a ojo, complementando los numeros de arriba.


In [ ]:
# Version agregada: mediana por epoca con banda intercuartilica, por dataset.
fig = P.trajectory_grid(traj, A.headline_columns(), color_by="dataset",
                        ncols=3, aggregate=True)
P.save(fig, "pilot-trayectorias")

# Version sin agregar: una linea por run. La banda esconde el run anomalo,
# asi que para inspeccion visual hace falta ver los individuales.
fig = P.trajectory_grid(traj, A.headline_columns(), color_by="dataset", ncols=3)
P.save(fig, "pilot-trayectorias-individuales");

## 7. Identidad GNS = M/m-coherence - 1

El barrido de sanidad encontro que dos de las ocho metricas **no son dos senales**: la escala de ruido simple es la m-coherencia reparametrizada, con $M$ el tamano del conjunto de medicion. Un coeficiente de correlacion de 1.00 seria compatible con cualquier relacion monotona; la diagonal solo la satisface la igualdad, asi que la comprobacion honesta es dibujar una contra la otra.

In [ ]:
M = 256   # tamano del conjunto de medicion (FIXED_KNOBS)
pred = M / traj["mcoh/global"] - 1
fig = P.identity_scatter(pred, traj["noise_scale/simple"],
                         xlabel="M / m-coherence - 1", ylabel="noise scale (simple)",
                         log=True)
P.save(fig, "pilot-identidad-gns-mcoh")
err = ((pred - traj["noise_scale/simple"]).abs() / traj["noise_scale/simple"].abs()).max()
print(f"error relativo maximo: {err:.2e}")

## 8. Coste de la instrumentacion

Cuanto cuesta medir por cada segundo de entrenamiento. La cifra absoluta no dice nada util porque las celdas difieren en dos ordenes de magnitud de coste; lo que decide si la instrumentacion completa es asumible es el **cociente**, y lo que lo gobierna es la arquitectura: el barrido per-sample es caro donde la ultima capa es grande. La linea marca la paridad, y a su derecha medir cuesta mas que entrenar.

In [ ]:
coste = summ.assign(ratio=summ["metric_seconds"] / summ["train_seconds"])
fig = P.strip(coste, "ratio", "model", order=["cnn", "resnet18", "fc"],
              xlabel="segundos de medicion por segundo de entrenamiento",
              reference=1.0, reference_label="paridad")
P.save(fig, "pilot-coste")
coste.groupby("model")["ratio"].describe()[["min", "50%", "max"]].round(2)

## 6. Sintesis

Apuntar aqui las conclusiones de "tienen sentido / hay que vigilar" tras correr las celdas: validez estructural, identidades, que metricas concuerdan con su teoria, cuales degeneran y los bloques de redundancia. Estas observaciones son **descriptivas** del pilot; ninguna decision confirmatoria se toma con estos datos.
